# 02 — Exploratory Data Analysis

Explore the trade data, market distribution, and baseline trader statistics.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path(".").resolve()))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_theme(style="whitegrid")

In [ ]:
# Load market index
idx = pd.read_parquet("data/processed/market_index.parquet")
print("Markets by category:")
print(idx.groupby(["category", "is_leaked"]).size().unstack(fill_value=0))

idx["date"] = pd.to_datetime(idx["news_timestamp"], unit="s")
print(f"
Date range: {idx['date'].min()} → {idx['date'].max()}")

In [ ]:
# Load filtered trades
trades = pd.read_parquet("data/processed/trades_filtered.parquet")
print(f"Total trades: {len(trades):,}")
print(f"Unique takers: {trades['taker'].nunique():,}")
print(f"Trades in leaked markets: {trades['is_leaked'].sum():,}")
print(trades.describe())

In [ ]:
# Volume distribution
fig, axes = plt.subplots(1, 2)
for ax, label, mask in [
    (axes[0], "Leaked", trades["is_leaked"]==1),
    (axes[1], "Control", trades["is_leaked"]==0)
]:
    data = trades.loc[mask, "usdc_amount"].clip(upper=5000)
    ax.hist(data, bins=50, color="crimson" if label=="Leaked" else "steelblue", alpha=0.7)
    ax.set_title(f"{label} market trade sizes")
    ax.set_xlabel("USDC")
plt.tight_layout()
plt.savefig("notebooks/fig_volume_dist.png", dpi=150)
plt.show()

In [ ]:
# Pre-news timing distribution
trades["mins_before_news"] = (trades["news_timestamp"] - trades["block_timestamp"]) / 60
fig, ax = plt.subplots()
for label, mask, color in [("Leaked", trades["is_leaked"]==1, "crimson"), ("Control", trades["is_leaked"]==0, "steelblue")]:
    d = trades.loc[mask, "mins_before_news"].clip(0, 48*60)
    ax.hist(d, bins=48, alpha=0.5, label=label, color=color, density=True)
ax.set_xlabel("Minutes before news")
ax.set_ylabel("Density")
ax.set_title("Trade timing relative to news release")
ax.legend()
plt.tight_layout()
plt.savefig("notebooks/fig_timing.png", dpi=150)
plt.show()

In [ ]:
# Direction bias comparison
if "D" in trades.columns:
    dir_col = "D"; buy_val = 1
else:
    dir_col = "taker_direction"; buy_val = "BUY"
buy_frac = trades.groupby(["taker","is_leaked"]).apply(lambda g: (g[dir_col]==buy_val).mean()).reset_index()
buy_frac.columns = ["taker","is_leaked","buy_fraction"]
fig, ax = plt.subplots()
for label, mask, c in [("Leaked",buy_frac["is_leaked"]==1,"crimson"),("Control",buy_frac["is_leaked"]==0,"steelblue")]:
    ax.hist(buy_frac.loc[mask,"buy_fraction"], bins=30, alpha=0.6, label=label, color=c, density=True)
ax.set_xlabel("Buy fraction")
ax.set_title("Direction bias by market type")
ax.legend()
plt.tight_layout()
plt.savefig("notebooks/fig_direction_bias.png", dpi=150)
plt.show()